In [1]:
import os
import json
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score
import numpy as np
import pandas as pd

def read_json(path):
    with open(path, 'r', encoding="utf-8") as f:
        data = json.load(f)
    return data

def write_json(data, path):
    if not os.path.exists(os.path.dirname(path)):
        os.makedirs(os.path.dirname(path))
    with open(path, 'w', encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

In [2]:
import numpy as np

def compute_forgetting_matrix(accuracy_matrix):
    """
    Computes forgetting for each task.
    
    Parameters:
    - accuracy_matrix: 2D numpy array of shape (num_tasks, num_tasks)
      Each row i represents accuracy on task i after training on tasks j=0..T-1
    
    Returns:
    - forgetting: list of forgetting values for each task
    - avg_forgetting: average forgetting across tasks (excluding the last task)
    """
    num_tasks = accuracy_matrix.shape[0]
    forgetting = []

    for i in range(num_tasks - 1):
        max_acc = np.max(accuracy_matrix[i, :i+1])  # Max accuracy before final training
        final_acc = accuracy_matrix[i, -1]          # Accuracy after training on all tasks
        forgetting.append(max_acc - final_acc)

    avg_forgetting = np.mean(forgetting)
    return forgetting, avg_forgetting


In [11]:
# matrices preparation on T5
def compute_forgetting_matrix_t5(input_folder, test_folder):
    results = []
    for i in range(1, 6):
       input_file = f"/Users/sefika/phd_projects/llm-catastrophic-re/results_all/last_results/corona/si/test/{input_folder}/task_{i}/test.json"
       start=0 
       predictions = read_json(input_file)
       predictions = [item['response'][0][0] for item in predictions]
       
       for id in range(1, i+1):
        test_file = f"/Users/sefika/phd_projects/llm-catastrophic-re/dataset/corona/test/{test_folder}/task_{id}/test.json"
        test_data = read_json(test_file)
        
        y_true = [item['relation'] for item in test_data]
        y_true = y_true[start:start+len(y_true)]
        print(f"Task {i} - Task {id} True Size: {len(y_true)}")
        y_task = predictions[start:start+len(y_true)]

        print(f"Task {i} - Task {id} Size: {len(y_task)}")
        acc = accuracy_score(y_true, y_task)
        print(f"Task {i} - Task {id} Accuracy: {acc:.4f}")


        row = {'base_task':i , 'task':id, 'accuracy':acc}
        print(row)
        start += len(y_true)
        results.append(row)
    return results
for run_id in range(1, 6):
    results = compute_forgetting_matrix_t5(
        input_folder=f"run_{run_id}",
        test_folder=f"run_{run_id}"
    )
    results
    write_json(results, f"/Users/sefika/phd_projects/llm-catastrophic-re/results_all/last_results/corona/si/model_{run_id}_forgetting_matrix.json")


Task 1 - Task 1 True Size: 210
Task 1 - Task 1 Size: 210
Task 1 - Task 1 Accuracy: 0.8571
{'base_task': 1, 'task': 1, 'accuracy': 0.8571428571428571}
Task 2 - Task 1 True Size: 210
Task 2 - Task 1 Size: 210
Task 2 - Task 1 Accuracy: 0.7381
{'base_task': 2, 'task': 1, 'accuracy': 0.7380952380952381}
Task 2 - Task 2 True Size: 45
Task 2 - Task 2 Size: 45
Task 2 - Task 2 Accuracy: 0.7556
{'base_task': 2, 'task': 2, 'accuracy': 0.7555555555555555}
Task 3 - Task 1 True Size: 210
Task 3 - Task 1 Size: 210
Task 3 - Task 1 Accuracy: 0.6143
{'base_task': 3, 'task': 1, 'accuracy': 0.6142857142857143}
Task 3 - Task 2 True Size: 45
Task 3 - Task 2 Size: 45
Task 3 - Task 2 Accuracy: 0.7556
{'base_task': 3, 'task': 2, 'accuracy': 0.7555555555555555}
Task 3 - Task 3 True Size: 4
Task 3 - Task 3 Size: 4
Task 3 - Task 3 Accuracy: 1.0000
{'base_task': 3, 'task': 3, 'accuracy': 1.0}
Task 4 - Task 1 True Size: 210
Task 4 - Task 1 Size: 210
Task 4 - Task 1 Accuracy: 0.5381
{'base_task': 4, 'task': 1, 'accu

/opt/anaconda3/lib/python3.11/site-packages/numpy/lib/function_base.py:520: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis, **keepdims_kw)
/opt/anaconda3/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/opt/anaconda3/lib/python3.11/site-packages/numpy/lib/function_base.py:520: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis, **keepdims_kw)
/opt/anaconda3/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Task 5 - Task 1 True Size: 1322
Task 5 - Task 1 Size: 1322
Task 5 - Task 1 Accuracy: 0.8298
{'base_task': 5, 'task': 1, 'accuracy': 0.829803328290469}
Task 5 - Task 2 True Size: 22
Task 5 - Task 2 Size: 22
Task 5 - Task 2 Accuracy: 1.0000
{'base_task': 5, 'task': 2, 'accuracy': 1.0}
Task 5 - Task 3 True Size: 162
Task 5 - Task 3 Size: 162
Task 5 - Task 3 Accuracy: 1.0000
{'base_task': 5, 'task': 3, 'accuracy': 1.0}
Task 5 - Task 4 True Size: 3
Task 5 - Task 4 Size: 3
Task 5 - Task 4 Accuracy: 1.0000
{'base_task': 5, 'task': 4, 'accuracy': 1.0}
Task 5 - Task 5 True Size: 56
Task 5 - Task 5 Size: 56
Task 5 - Task 5 Accuracy: 0.7500
{'base_task': 5, 'task': 5, 'accuracy': 0.75}


In [ ]:
# matrices preparation on Mistral

In [ ]:
# matrices preparation on Llama-2